# 01 · Data preparation and EDA

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = ""
USE_DRIVE = True

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub", "scikit-learn", "scipy", "pandas", "matplotlib", "pillow", "tqdm"], check=True)
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        os.environ.setdefault("XRAY_ARTIFACTS", "/content/drive/MyDrive/xray_project/artifacts")
        os.environ.setdefault("XRAY_RESULTS", "/content/drive/MyDrive/xray_project/results")


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "config.py").exists():
            return p


ROOT = find_root()
if ROOT is None:
    assert REPO_URL, "Set REPO_URL or open the notebook from inside the repository"
    subprocess.run(["git", "clone", REPO_URL, "xray-vision-pneumonia-CNN"], check=True)
    ROOT = Path("xray-vision-pneumonia-CNN").resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from src import config as C
from src import dataset as D
from src import eda
from src.utils import save_json, sync_results_to_repo, write_env

C.ensure_dirs()
write_env()
root = D.find_data_root()
print(root)

## Manifest, de-duplication and patient-grouped split

In [ ]:
manifest, audit = D.build_manifest(root)
D.save_manifest(manifest)
save_json(audit, C.AUDIT_JSON)
print(json.dumps(audit, indent=2))

In [ ]:
counts = pd.DataFrame({s: audit["counts"][s] for s in ("train", "val", "test")}).T
counts["pneumonia_per_normal"] = counts["PNEUMONIA"] / counts["NORMAL"]
counts.to_csv(C.RESULTS_DIR / "data_counts.csv")
official = pd.DataFrame(audit["official_counts"]).T
official.to_csv(C.RESULTS_DIR / "data_counts_official.csv")
display(official)
display(counts)

In [ ]:
eda.plot_counts(counts, C.FIG_DIR / "eda_counts.png")

## Image statistics

In [ ]:
df = D.load_manifest(root)
df = df[df["split"].isin(["train", "val", "test"])].reset_index(drop=True)
stats, thumbs = eda.load_or_compute_stats(df)
stats.head()

In [ ]:
eda.plot_samples(stats, thumbs, C.FIG_DIR / "eda_samples.png")
eda.plot_intensity_per_class(stats, thumbs, C.FIG_DIR / "eda_intensity_per_class.png")
eda.plot_mean_images(stats, thumbs, C.FIG_DIR / "eda_mean_images.png")

In [ ]:
eda.plot_sizes(stats, C.FIG_DIR / "eda_sizes.png")
display(pd.crosstab(stats["label"], stats["mode"]))
display(stats.groupby(["split", "label"])[["width", "height", "aspect", "file_bytes"]].agg(["mean", "std"]).round(2))

In [ ]:
brightness = stats.groupby(["split", "label"])[["mean", "std", "p1", "p99"]].mean().round(2)
brightness.to_csv(C.RESULTS_DIR / "eda_brightness.csv")
brightness

In [ ]:
eda.plot_marker_gallery(stats, thumbs, C.FIG_DIR / "eda_markers.png")
marker = stats.groupby("label")["marker_score"].describe()
marker.to_csv(C.RESULTS_DIR / "eda_marker_scores.csv")
marker

## Shortcut and distribution-shift checks

In [ ]:
meta = eda.metadata_shortcut(stats)
meta.to_csv(C.RESULTS_DIR / "eda_metadata_shortcut.csv", index=False)
meta

In [ ]:
adv = eda.adversarial_validation(stats, thumbs)
adv.to_csv(C.RESULTS_DIR / "eda_adversarial_validation.csv", index=False)
adv

In [ ]:
sync_results_to_repo()